# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 6: Interactive Visual Analytics with Folium
**Author: Gautam825406**

In [ ]:
!pip install folium pandas numpy --quiet

In [ ]:
import folium
import pandas as pd
import numpy as np
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon
import math

print('Folium version:', folium.__version__)

## Load SpaceX Launch Data

In [ ]:
spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
print('Shape:', spacex_df.shape)
spacex_df.head(5)

In [ ]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
print('Launch sites:')
launch_sites_df

## TASK 1: Mark all launch sites on a map

In [ ]:
# Start location: NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add circle and label for each launch site
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(
    folium.Popup('NASA Johnson Space Center'))
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0),
    html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',))
site_map.add_child(circle)
site_map.add_child(marker)

for idx, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    circle = folium.Circle(coordinate, radius=1000, color='#000080', fill=True).add_child(
        folium.Popup(row['Launch Site']))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#000080;"><b>%s</b></div>' % row['Launch Site']))
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

## TASK 2: Mark success/failed launches for each site on the map

In [ ]:
spacex_df.tail(20)

In [ ]:
marker_cluster = MarkerCluster()

def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

site_map2 = folium.Map(location=nasa_coordinate, zoom_start=5)
site_map2.add_child(marker_cluster)

for idx, row in spacex_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    marker_color = assign_marker_color(row['class'])
    cluster_marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(color=marker_color, icon='info-sign'),
        popup=folium.Popup(f"Site: {row['Launch Site']}<br>Outcome: {'Success' if row['class']==1 else 'Failure'}", parse_html=True)
    )
    marker_cluster.add_child(cluster_marker)

site_map2

## TASK 3: Calculate distances between launch site and its proximities

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1_r = math.radians(lat1)
    lat2_r = math.radians(lat2)
    dLat = math.radians(lat2 - lat1)
    dLon = math.radians(lon2 - lon1)
    a = math.sin(dLat/2)**2 + math.cos(lat1_r)*math.cos(lat2_r)*math.sin(dLon/2)**2
    c = 2*math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

print('Distance function defined')

In [ ]:
# CCAFS LC-40 coordinates
launch_site_lat = 28.562302
launch_site_lon = -80.577356

# Nearby locations
nearby_coords = [
    {'name': 'Coastline', 'lat': 28.56318, 'lon': -80.56802},
    {'name': 'City (Cocoa, FL)', 'lat': 28.3861, 'lon': -80.7428},
    {'name': 'Highway US-1', 'lat': 28.5671, 'lon': -80.5778},
    {'name': 'Railway', 'lat': 28.5721, 'lon': -80.5785},
]

for loc in nearby_coords:
    dist = calculate_distance(launch_site_lat, launch_site_lon, loc['lat'], loc['lon'])
    print(f"Distance to {loc['name']}: {dist:.2f} km")

In [ ]:
# Draw proximity lines on map
site_map3 = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=12)

# Launch site marker
folium.Marker(
    [launch_site_lat, launch_site_lon],
    popup='CCAFS LC-40 Launch Site',
    icon=folium.Icon(color='blue', icon='rocket', prefix='fa')
).add_to(site_map3)

colors = ['red', 'green', 'blue', 'purple']
for i, loc in enumerate(nearby_coords):
    dist = calculate_distance(launch_site_lat, launch_site_lon, loc['lat'], loc['lon'])
    folium.Marker([loc['lat'], loc['lon']], popup=f"{loc['name']}\nDist: {dist:.2f} km").add_to(site_map3)
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [loc['lat'], loc['lon']]],
        color=colors[i],
        weight=2,
        popup=f"{dist:.2f} km to {loc['name']}"
    ).add_to(site_map3)

site_map3

In [ ]:
print('Folium Interactive Map Analysis Complete')
print('Key findings:')
print('- All launch sites are near the coastline (within 1km)')
print('- Sites are 20-60 km from major cities')
print('- Good highway and railway access within 5km')
print('- KSC and CCAFS sites are clustered together in Florida')